# Analyze a trained quantizer

Load a completed quantization run to inspect training history, code usage, collisions,
reconstruction, and similarity within Semantic ID prefixes. Training and mapping generation
live in `scripts/train_quantizer.py`; this notebook does not modify the run.

Run the cells in order using the project's Python environment. Set `RUN_DIR` below.
The kernel can start in the repository root or in `notebooks/`. Saved configuration paths
are interpreted relative to the repository root, just as they are in the CLI scripts.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

from generative_recs.config import load_config
from generative_recs.rqvae import RQVAE
from generative_recs.utils import get_device, set_seed

In [ ]:
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RUN_DIR = PROJECT_ROOT / "outputs/toys/baseline/quantization"

config = load_config(RUN_DIR / "configs/quantization.yaml", "quantization")
DATA_DIR = PROJECT_ROOT / config.preparation_dir

set_seed(config.seed)
device = get_device()

embeddings_dict = torch.load(DATA_DIR / "embeddings.pt", map_location="cpu", weights_only=True)
mappings = torch.load(RUN_DIR / "mappings.pt", map_location="cpu", weights_only=True)
asin_to_sid = mappings["asin_to_sid"]
asins = list(asin_to_sid)
embedding_matrix = torch.stack([embeddings_dict[asin] for asin in asins]).float()
all_indices = torch.tensor([asin_to_sid[asin][:-1] for asin in asins])

model = RQVAE(**config.model.model_dump()).to(device)
model.load_state_dict(torch.load(RUN_DIR / "rqvae_best.pt", map_location=device, weights_only=True))
model.eval()

items_df = pd.read_parquet(DATA_DIR / "items.parquet")
asin_to_text = dict(zip(items_df["asin"], items_df["text"], strict=True))
print(f"Loaded {len(asins):,} items from {RUN_DIR}")

## Training history and code usage

Catalog reconstruction is measured on the full item catalog at each evaluation step.
The selected checkpoint has the lowest catalog reconstruction loss.

Normalized entropy measures how evenly assignments are distributed across the codebook.
The active-code fraction below measures how many codes are used at least once; these are
different measurements. Neither is sufficient to establish recommendation quality.

In [ ]:
history = pd.read_csv(RUN_DIR / "training_history.csv")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

history.plot(x="step", y=["train_recon_loss", "eval_recon_loss"], ax=axes[0])
axes[0].set_title("Reconstruction loss")
axes[0].set_ylabel("1 − cosine similarity")

history.plot(x="step", y="train_commit_loss", ax=axes[1])
axes[1].set_title("Commitment loss")

entropy_cols = [name for name in history.columns if name.startswith("eval_normalized_entropy_level_")]
history.plot(x="step", y=entropy_cols, ax=axes[2])
axes[2].set_title("Normalized codebook entropy")
axes[2].set_ylabel("Normalized entropy")
axes[2].set_ylim(0, 1.05)
fig.tight_layout()

best = history.loc[history["eval_recon_loss"].idxmin()]
print(f"Selected step: {int(best['step']):,}; reconstruction loss: {best['eval_recon_loss']:.6f}")
active_codes = [len(all_indices[:, level].unique()) for level in range(config.model.num_quantizers)]
display(
    pd.DataFrame(
        {
            "level": range(config.model.num_quantizers),
            "active_codes": active_codes,
            "active_fraction": [count / config.model.codebook_size for count in active_codes],
        }
    )
)

## Collisions

The final token distinguishes items that share the same quantized prefix. Group by the
preceding tokens to see how often this happens, then inspect some example groups.

In [ ]:
sid_to_items: dict[tuple[int, ...], list[str]] = {}
for asin, sid in asin_to_sid.items():
    sid_to_items.setdefault(sid[:-1], []).append(asin)

collision_sizes = Counter(len(items) for items in sid_to_items.values())
print(f"Unique quantized prefixes: {len(sid_to_items):,}")
print(
    f"Groups with collisions: {sum(count for size, count in collision_sizes.items() if size > 1):,}"
)
colliding_items = sum(size * count for size, count in collision_sizes.items() if size > 1)
print(f"Items in collision groups: {colliding_items:,}")
display(pd.DataFrame(sorted(collision_sizes.items()), columns=["group_size", "number_of_groups"]))

In [ ]:
# Show items that collide (share the same quantized prefix)
collisions = {sid: items for sid, items in sid_to_items.items() if len(items) > 1}
print(f"Found {len(collisions)} colliding Semantic IDs\n")

# Show up to 5 collision groups
for prefix, items in list(collisions.items())[:5]:
    print(f"Semantic ID: {prefix}")
    for asin in items:
        text = asin_to_text.get(asin, "N/A")
        # Show first 120 chars of item text
        print(f"  [{asin}] {text[:120]}...")
    print()

## Prefix neighbors

Inspect items sharing the first codeword and then longer prefixes. Similar-looking items
would support the usefulness of the grouping, but codes are not guaranteed category labels.

In [ ]:
for prefix_length in range(1, config.model.num_quantizers):
    groups: dict[tuple[int, ...], list[str]] = {}
    for asin, sid in asin_to_sid.items():
        groups.setdefault(sid[:prefix_length], []).append(asin)

    examples = [(prefix, items) for prefix, items in groups.items() if len(items) >= 2][:3]
    print(f"Shared prefix length: {prefix_length}")
    for prefix, items in examples:
        print(f"  {prefix}: {len(items)} items")
        for asin in items[:5]:
            print(f"    [{asin}] {asin_to_text.get(asin, 'N/A')[:120]}")

## Reconstruction quality

Compare reconstructed embeddings with their original values. Also compare every item
with the same average embedding: a high cosine score can partly reflect similarities
already present throughout the catalog. This is a diagnostic, not a retrieval metric.

In [ ]:
cos_sims = []
with torch.no_grad():
    for batch in embedding_matrix.split(config.training.batch_size):
        batch = batch.to(device)
        reconstructed, _, _, _ = model(batch)
        cos_sims.append(F.cosine_similarity(batch, reconstructed, dim=1).cpu())
cos_sims = torch.cat(cos_sims)
mean_embedding = embedding_matrix.mean(dim=0, keepdim=True)
mean_baseline = F.cosine_similarity(embedding_matrix, mean_embedding, dim=1)

display(
    pd.DataFrame(
        {
            "Quantizer": cos_sims.numpy(),
            "Average embedding": mean_baseline.numpy(),
        }
    ).describe()
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(cos_sims.numpy(), bins=50, alpha=0.6, label="Quantizer")
ax.hist(mean_baseline.numpy(), bins=50, alpha=0.6, label="Average embedding")
ax.set_xlabel("Cosine similarity with the original embedding")
ax.set_ylabel("Items")
ax.set_title("Reconstruction quality distribution")
ax.legend()
fig.tight_layout()

## Similarity within shared prefixes

Compare pairs of distinct items from the catalog with pairs drawn from shared-prefix groups.
For small groups, calculate all pairs; for larger groups, sample pairs using a fixed seed.
Each eligible group has equal weight in the reported mean. Singleton groups are excluded,
and the table shows how many groups and items contribute at each level.

Increasing similarity with longer prefixes is useful evidence of semantic grouping, but it
is not guaranteed. The participating groups change at each level, and this analysis cannot
replace evaluating the retrieval model.

In [ ]:
def avg_pairwise_cosine_sim(embeddings: torch.Tensor, max_pairs: int = 5000) -> float:
    """Mean cosine similarity of distinct pairs, sampling when a group exceeds 200 items."""
    n = len(embeddings)
    if n < 2:
        return float("nan")

    normed = F.normalize(embeddings, dim=1)
    if n <= 200:
        sim_matrix = normed @ normed.T
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        return sim_matrix[mask].mean().item()

    idx1 = torch.randint(0, n, (max_pairs,))
    # A nonzero offset excludes self-pairs without discarding samples.
    idx2 = (idx1 + torch.randint(1, n, (max_pairs,))) % n
    return (normed[idx1] * normed[idx2]).sum(dim=1).mean().item()


set_seed(config.seed)
results = [
    {
        "grouping": "Random catalog pairs",
        "mean_cosine": avg_pairwise_cosine_sim(embedding_matrix),
        "groups": 1,
        "items": len(asins),
    }
]

for prefix_length in range(1, config.model.num_quantizers + 1):
    groups: dict[tuple[int, ...], list[int]] = {}
    for index, asin in enumerate(asins):
        prefix = asin_to_sid[asin][:prefix_length]
        groups.setdefault(prefix, []).append(index)

    eligible = [indices for indices in groups.values() if len(indices) >= 2]
    scores = [avg_pairwise_cosine_sim(embedding_matrix[indices]) for indices in eligible]
    results.append(
        {
            "grouping": f"Shared prefix of length {prefix_length}",
            "mean_cosine": sum(scores) / len(scores) if scores else float("nan"),
            "groups": len(eligible),
            "items": sum(len(indices) for indices in eligible),
        }
    )

display(pd.DataFrame(results))